# 10. Logistic Regression

Binary linear classifier with probabilities (`predict_proba`).
Tune regularization `C` with stratified CV (F1); keep max_iter=1000.

In [3]:
import sys
sys.path.append("..")

from sklearn.linear_model import LogisticRegression

from src.data import load_adult
from src.preprocess import split_features_target, make_model_pipeline
from src.evaluate import cross_val_scores, evaluate_on_test, save_results

X, y = split_features_target(load_adult("train"))
X_test, y_test = split_features_target(load_adult("test"))

print(X.shape, y.mean())

(32561, 12) 0.2408095574460244


In [5]:
candidates = [0.01, 0.1, 1, 10, 100]
selection_metric = "f1"

cv_by_C = {}
best_C = None
best_score = -1.0

for C in candidates:
    model = LogisticRegression(C=C, max_iter=1000, random_state=42)
    pipe = make_model_pipeline(model)
    cv = cross_val_scores(pipe, X, y)
    cv_by_C[C] = cv
    score = cv[selection_metric]["mean"]
    print(f"C={C}  CV {selection_metric}={score:.4f}  (±{cv[selection_metric]['std']:.4f})")

    if score > best_score:
        best_score = score
        best_C = C

print("selected C:", best_C)

C=0.01  CV f1=0.6327  (±0.0074)
C=0.1  CV f1=0.6451  (±0.0078)
C=1  CV f1=0.6476  (±0.0083)
C=10  CV f1=0.6482  (±0.0082)
C=100  CV f1=0.6478  (±0.0078)
selected C: 10


In [6]:
final_model = LogisticRegression(C=best_C,max_iter=1000, random_state=42)

cv = cv_by_C[best_C]
pipe2 = make_model_pipeline(final_model)
test = evaluate_on_test(pipe2, X, y, X_test, y_test)

path = save_results(
    "logistic_regression",
    cv,
    test,
    extra={
        "C": best_C,
        "max_iter": 1000,
        "candidates": candidates,
        "selection_metric": selection_metric,
        "cv_by_C": {
            str(c): {m: v["mean"] for m, v in summary.items()}
            for c, summary in cv_by_C.items()
        },
    },
)

print("Test:", test["metrics"])
print("Confusion matrix:", test["confusion_matrix"])
print("saved:", path)


Test: {'accuracy': 0.847122412628217, 'balanced_accuracy': 0.758670296148396, 'f1': 0.6461975835110163, 'roc_auc': 0.8986982229590753, 'average_precision': 0.7360911162533845}
Confusion matrix: [[11519, 916], [1573, 2273]]
saved: /Users/kang/Desktop/ML project/Estimating-Income/results/logistic_regression.json
